## Reinforcement Finetuning of Retail Agent

Finetune a Retail Agent with Reinforcement learning


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### Data

In [5]:
import json

with open("data/retail_agent_policy.md") as f:
    policy = f.read()

system_prompt_no_policy = "You are a helpful Retail Support Agent that helps customers with their orders, exchanges, returns and refunds."
system_prompt_with_policy = system_prompt_no_policy + "\n\n" + policy

with open("data/retail_agent_tools.json") as f:
    tool_definitions = json.load(f)

In [6]:
with open("data/retail_agent_partial_eval.jsonl") as partial_file, \
        open("data/retail_agent_data_eval_no_policy.jsonl", "w") as eval_no_policy_file, \
        open("data/retail_agent_data_eval_with_policy.jsonl", "w") as eval_with_policy_file:
    for line in partial_file:
        item = json.loads(line)
        item["tools"] = tool_definitions

        item["messages"][0]["content"] = system_prompt_no_policy
        eval_no_policy_file.write(json.dumps(item) + "\n")

        item["messages"][0]["content"] = system_prompt_with_policy
        eval_with_policy_file.write(json.dumps(item) + "\n")

### Grader

In [8]:
import json
import os
import requests

with open("tool_call_grader/tool_call_grader.py", "r") as f:
    grader_source = f.read()

grader = {
    "type": "python",
    "source": grader_source,
    "pass_threshold": 0.5,
}

headers = {
    "Authorization": f"Bearer {os.environ['AZURE_OPENAI_API_KEY']}",
}

validate_response = requests.post(os.environ['AZURE_OPENAI_ENDPOINT'] + "/openai/v1/fine_tuning/alpha/graders/validate",
    headers=headers, json={ "grader": grader })

print(f"Status: {validate_response.status_code}")
print(json.dumps(validate_response.json(), indent=2))

Status: 200
{
  "grader": {
    "type": "python",
    "source": "import json\nfrom collections import Counter\nfrom typing import Any, Dict, List\n\nfrom rouge_score import rouge_scorer\n\n\nclass GraderConfig:\n    include_tools: List[str] = []\n    \"\"\"include_tools can be specified to grade only certain tools (by function name). Otherwise grader will grade all calls.\"\"\"\n\n\ndef grade(sample: Dict, item: Dict) -> float:\n    config = GraderConfig()\n    return grade_with_config(sample, item, config)\n\n\ndef grade_with_config(sample: Dict, item: Dict, config: GraderConfig) -> float:\n    actual_calls = sample.get(\"output_tools\") or []\n    expected_calls = item[\"reference_tool_calls\"]\n\n    if not actual_calls:  # just for /graders/run API, load from output_text since /graders/run API doesn't support output_tools yet\n        try:\n            actual_calls = json.loads(sample[\"output_text\"])[\"output_tools\"]\n        except (KeyError, json.JSONDecodeError):\n           

In [4]:

run_payload = {
    "grader": grader,
    "model_sample": json.dumps({
        "output_tools": [
            { "id": "call_1", "type": "function", "function": { "name": "find_user_id_by_email", "arguments": json.dumps({"email": "user@example.com"}) } }
        ],
    }),
    "item": {
        "reference_tool_calls": [
            { "id": "call_1", "type": "function", "function": { "name": "find_user_id_by_email", "arguments": json.dumps({"email": "anotheruser@example.com"}) } }
        ],
    },
}

run_response = requests.post(os.environ['AZURE_OPENAI_ENDPOINT'] + "/openai/v1/fine_tuning/alpha/graders/run",
    headers=headers, json=run_payload)

print(f"Status: {run_response.status_code}")
print(run_response.json())

Status: 200
{'reward': 0.5, 'metadata': {'name': 'grader-X2v4CTKlPato', 'type': 'python', 'errors': {'formula_parse_error': False, 'sample_parse_error': False, 'sample_parse_error_details': None, 'truncated_observation_error': False, 'unresponsive_reward_error': False, 'invalid_variable_error': False, 'invalid_variable_error_details': None, 'other_error': False, 'python_grader_server_error': False, 'python_grader_server_error_type': None, 'python_grader_runtime_error': False, 'python_grader_runtime_error_details': None, 'model_grader_server_error': False, 'model_grader_refusal_error': False, 'model_grader_refusal_error_details': None, 'model_grader_parse_error': False, 'model_grader_parse_error_details': None, 'model_grader_exceeded_max_tokens_error': False, 'model_grader_server_error_details': None, 'endpoint_grader_internal_error': False, 'endpoint_grader_internal_error_details': None, 'endpoint_grader_server_error': False, 'endpoint_grader_server_error_details': None, 'endpoint_grad

### Reinforcement FineTuning Job

In [10]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"] + "/openai/v1/",
)

In [6]:
with open("data/reinforcement_retail_agent_train.jsonl", "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")

print(train_file.id)

file-32a1d9890c6845d593a61ce30f8c0e22


In [7]:
VALIDATION_FILE = "data/reinforcement_retail_agent_valid.jsonl"

with open("data/reinforcement_retail_agent_eval.jsonl") as eval_fp, \
        open(VALIDATION_FILE, "w") as valid_fp:
    eval_lines = eval_fp.readlines()
    valid_fp.writelines(eval_lines[:10])  # keep validation set small for quick evaluations

with open(VALIDATION_FILE, "rb") as f:
    valid_file = client.files.create(file=f, purpose="fine-tune")

print(valid_file.id)

file-33287a58c9f74565ad3a6870cf98df14


In [8]:
from mcp_server.mcp_server import tool_endpoints

tools = tool_endpoints()  # be careful to not print tools as it contains mcp api key

In [9]:
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=valid_file.id, 
    model="o4-mini",
    suffix="retail-agent",
    method=dict(
        type="reinforcement",
        reinforcement=dict(
            grader=grader,
            response_format=None,
            tools=tools,
            hyperparameters=dict(
                n_epochs=1,
                reasoning_effort="low",

                # eval_interval is number of training steps between evaluation runs.
                # Lower value means more frequent evaluations and more visibility into the learning process, but also longer training time.
                eval_interval=3,

                # eval_samples is number of evaluation samples to generate for every sample in validation file.
                # Every evaluation run will have # of samples = (# of validation file samples * eval_samples)
                # Higher value makes the validation curves more robust especially if there's stochasticity in model's outputs.
                # Evaluation score is averaged across multiple samples to reduce noise and reveal consistent patterns of learning.
                eval_samples=2,

                # compute_multiplier determines how hard should the model explore search space.
                # Higher value means more rollouts and diverse exploration.
                compute_multiplier=1.0,
            )
        )
    )
)

print(job.id)

HTTP Request: POST https://omi-ignite-demo-resource.openai.azure.com//openai/v1/fine_tuning/jobs "HTTP/1.1 201 Created"


ftjob-e453e3eb901e48e1bcf7aff170a761c1


In [20]:
import time
from datetime import datetime
from IPython.display import display, clear_output

try:
    while True:
        events = client.fine_tuning.jobs.list_events(job.id)
        clear_output(wait=True)
        print("Latest job events:")
        for event in events.data[-3:]:
            local_time = datetime.fromtimestamp(event.created_at).strftime("%Y-%m-%d %H:%M:%S")
            print("-", local_time, event.message)
        time.sleep(10)
except KeyboardInterrupt:
    pass

Latest job events:
- 2026-04-05 20:14:35 Preprocessing completed for file training file.
- 2026-04-05 20:14:34 Preprocessing running for file training file.
- 2026-04-05 20:12:18 Job enqueued. Waiting for jobs ahead to complete.


### Evaluation

- https://developers.openai.com/cookbook/examples/evaluation/use-cases/mcp_eval_notebook
- https://developers.openai.com/cookbook/examples/evaluation/use-cases/completion-monitoring

In [20]:
def load_eval_data(path):
    items = []
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            items.append({
                "input_messages": d["messages"],
                "reference_tool_calls": d["reference_tool_calls"],
                "reference_reply": d.get("reference_reply", ""),
            })
    return items

eval_no_policy = load_eval_data("data/retail_agent_data_eval_no_policy.jsonl")
eval_with_policy = load_eval_data("data/retail_agent_data_eval_with_policy.jsonl")

In [21]:
retail_agent_eval = client.evals.create(
    name="Retail Agent Eval",
    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "input_messages": {"type": "array"},
                "reference_tool_calls": {"type": "array"},
                "reference_reply": {"type": "string"},
            },
        },
        "include_sample_schema": True,
    },
    testing_criteria=[{
        "type": "python",
        "name": "Tool Call Grader",
        "source": grader_source,
        "pass_threshold": 0.7,
    }],
)

eval_id = retail_agent_eval.id
print(eval_id)

eval_69db911bbfa88191b9c3ab66d174b497


In [ ]:
mcp_tool = {
    "type": "mcp",
    "server_label": "retail_mcp",
    "server_url": os.environ["MCP_ENDPOINT"],
    "headers": {
        "X-MCP-API-Key": os.environ["MCP_API_KEY"],
        "X-MCP-Read-Only": "true",
    },
    "require_approval": "never",
}

TODO: convert input_messages to responsesapi format

def create_run(name, items):
    return client.evals.runs.create(
        name=name,
        eval_id=eval_id,
        data_source={
            "type": "responses",
            "source": {"type": "file_content", "content": [{"item": item} for item in items]},
            "input_messages": {
                "type": "item_reference",
                "item_reference": "item.input_messages",
            },
            "model": "o4-mini",
            "sampling_params": {
                "max_completions_tokens": 10000,
                "reasoning_effort": "low",
                "tools": [mcp_tool],
            },
        },
    )

run_no_policy = create_run("o4-mini (no policy)", eval_no_policy)
run_with_policy = create_run("o4-mini (with policy)", eval_with_policy)

print(run_no_policy.id, run_with_policy.id)

evalrun_69db92a54a8881919f83304084d755bc evalrun_69db92a6f47c819186584d9701c4ca79


In [24]:
import time
from IPython.display import display, clear_output

def poll_runs(eval_id, run_ids):
    while True:
        runs = [client.evals.runs.retrieve(rid, eval_id=eval_id) for rid in run_ids]
        clear_output(wait=True)
        for run in runs:
            print(run.name, run.status, run.result_counts)
        if all(run.status in {"completed", "failed"} for run in runs):
            break
        time.sleep(20)

eval_id = retail_agent_eval.id
poll_runs(eval_id, [run_no_policy.id, run_with_policy.id])

o4-mini (no policy) failed ResultCounts(errored=0, failed=0, passed=0, total=0)
o4-mini (with policy) failed ResultCounts(errored=0, failed=0, passed=0, total=0)


In [ ]:
for run in [run_no_policy, run_with_policy]:
    output = client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_def.id)
    print(f"\n# {run.name}")
    for item in output:
        print(f"  Status: {item.status}, Score: {item.sample.score}")